# Detecção de Duplicatas

Detecção de pares semanticamente semelhantes usando embeddings e similaridade de cosseno.

In [ ]:

import json, re, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

with open("manifestacoes.json", encoding="utf-8") as f:
    dados = json.load(f)
df = pd.DataFrame(dados)
df["texto"] = df["texto"].fillna("")
textos = df["texto"].tolist()
ids = df["id"].tolist()
print(df.shape)
df.head()


In [ ]:

from sentence_transformers import SentenceTransformer
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
E = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=True)
SIM = cosine_similarity(E)


## Função exigida

In [ ]:

def detectar_duplicatas(textos, limiar=0.85):
    emb = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=False)
    sim = cosine_similarity(emb)
    pares = []
    for i in range(len(textos)):
        for j in range(i+1, len(textos)):
            if sim[i,j] >= limiar:
                pares.append({
                    "id_1": ids[i], "id_2": ids[j],
                    "similaridade": float(sim[i,j]),
                    "texto_1": textos[i], "texto_2": textos[j]
                })
    return pd.DataFrame(pares).sort_values("similaridade", ascending=False).reset_index(drop=True)

duplicatas = detectar_duplicatas(textos, 0.85)
duplicatas


## Matriz de similaridade — heatmap

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(13,10))
sns.heatmap(SIM, xticklabels=ids, yticklabels=ids, cmap="viridis", vmin=0, vmax=1)
plt.title("Similaridade semântica entre manifestações")
plt.xticks(rotation=90, fontsize=6); plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()


## Avaliação

O enunciado informa aproximadamente 15% de duplicatas semânticas, mas não fornece uma lista explícita de pares-verdade no conjunto recebido. Assim, o notebook apresenta os pares detectados pelo limiar e permite inspeção manual. Falsos positivos e negativos devem ser registrados após essa conferência. O limiar 0,85 é o valor solicitado no desafio; ele pode ser ajustado e comparado com percentis da distribuição.

In [ ]:

tri = SIM[np.triu_indices_from(SIM, k=1)]
print("Percentil 90:", np.percentile(tri,90))
print("Percentil 95:", np.percentile(tri,95))
print("Quantidade acima de 0.85:", int((tri>=0.85).sum()))
